# PokeZero playground

A hands-on tour of the repo: step through a game, pit agents against each other, look inside the
encoder and network, edit the config and run a short training loop, then watch a replay.
Works locally (Jupyter / VS Code) and on Google Colab.

**Why is the game CPU-only, and is a GPU best for training?**

* The *game engine* (`kaggle_environments.envs.cabt`) is a native library that only runs on the
  CPU. Every self-play game, and every search simulation, is CPU work. That is the bottleneck.
* The *network* can use a GPU. `TrainConfig.device` (`cpu` / `cuda` / `mps` / `auto`) moves the
  gradient steps there; self-play and the submission agent stay on CPU.
* With a network this small, a GPU is mostly idle and waiting on the engine, so more CPU cores buy
  more games per dollar. A GPU starts to pay off with a much bigger network, batched inference
  across many parallel games, or behaviour cloning from big replay exports.

The notebook is saved without outputs. It prints option *types* and shapes, never card names or text.

## How to run this visually

Run one cell at a time and read its output before moving on. **Shift+Enter** runs a cell and moves
to the next; do not "Run all" the first time.

* **Google Colab (no install):** open https://colab.research.google.com, choose *GitHub*, paste
  `dhruvcharan13/PokeZero`, open `notebooks/playground.ipynb`. (Private repo: sign in to GitHub in
  the dialog, or upload the file.) Runtime > Change runtime type > GPU is optional; see section 4.
* **VS Code:** install the Python and Jupyter extensions, open the repo, `uv sync --extra notebook`,
  open the `.ipynb`, pick the `.venv` kernel, then click the play icon on each cell.
* **Jupyter in a browser:** `uv sync --extra notebook && uv run jupyter lab`, open `notebooks/playground.ipynb`.

Each section has a **Why** box listing the design decisions in it and where they live in the code,
so you can explain every choice. Sections 1 to 5 follow one game from raw engine data to a trained
network; section 6 collects the decisions in one table.

## 0. Setup

In [ ]:
import sys

if "google.colab" in sys.modules:
    # Colab: install the package from GitHub (a private repo needs a token in the URL).
    %pip install -q "pokezero @ git+https://github.com/dhruvcharan13/PokeZero.git"

In [ ]:
import json
import random
import tempfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from pokezero.agents import HeuristicAgent, RandomAgent
from pokezero.agents.az import PolicyAgent
from pokezero.config import load_config, to_dict
from pokezero.env.cabt import CabtEnv
from pokezero.eval.arena import format_report, play_match, round_robin
from pokezero.model.encoding import encode
from pokezero.model.net import NetConfig, PokeNet
from pokezero.train.learner import resolve_device

DEVICE = resolve_device("auto")  # cuda > mps > cpu
print("training device:", DEVICE, "| game engine: always CPU")

## 1. Step through a game

`CabtEnv.reset()` returns a `Decision`: who is acting, what kind of prompt it is, how many options
they must pick (`min_count`..`max_count`), and the observation that player sees (their own hand
plus the public board; the opponent's hand is hidden).

**Why** (`env/cabt.py`, `env/base.py`)

* The engine is a compiled library called through `ctypes`; we wrap it in `CabtEnv` so everything
  else only sees a `Decision` (competition-neutral). Swapping the game or mocking it touches one file.
* A game is a stream of *questions*: the engine pauses whenever one player must choose, and the
  answer is a list of option indices. That is why every agent is just `act(decision) -> list[int]`.
* Picks are validated before reaching the engine (`validate_picks`): an illegal move fails loudly in
  Python instead of corrupting the native game.
* The engine cannot be seeded, so games are made reproducible by recording every pick (`Trajectory`),
  not by re-running with a seed. `seed` only seeds our own Python RNG.
* The observation holds only what the acting player may see: their own hand and the public board.

In [ ]:
env = CabtEnv()
d = env.reset(seed=0)
print("acting player:", d.player, "| select_type:", d.select_type, "| context:", d.context)
print("pick", d.min_count, "to", d.max_count, "of", d.num_options, "options")
print("observation keys:", sorted(d.obs))
print("current keys:", sorted(d.obs["current"]))
print("option types:", Counter(op["type"] for op in d.options))
print("first option:", d.options[0])

In [ ]:
# Play the first 15 decisions by hand (random legal picks) and watch the prompt change.
rng = random.Random(1)
for step in range(15):
    k = rng.randint(d.min_count, min(d.max_count, d.num_options))
    picks = rng.sample(range(d.num_options), k)
    kinds = sorted({op["type"] for op in d.options})
    print(
        f"{step:2d} player={d.player} select={d.select_type} ctx={d.context:2d} "
        f"n={d.num_options:2d} pick {picks} option types={kinds}"
    )
    nxt = env.step(picks)
    if nxt is None:
        break
    d = nxt
env.close()

## 2. Agents against each other

`RandomAgent`, `HeuristicAgent` (hand-written rules) and `PolicyAgent` (a network, here untrained).
`play_match` alternates seats. Try changing the game counts, or write your own agent: anything with
`reset()` and `act(decision) -> list[int]` works.

**Why** (`agents/`, `eval/arena.py`)

* Baselines first: random is the floor and a hand-written heuristic (play cards, then best attack,
  end turn last) is a sanity bar. If a trained net can't beat these, something is broken.
* The heuristic falls back to random if it ever errors, so it can never submit an illegal move.
* `play_match` alternates seats, because going first may matter; otherwise win rate is biased.
* The round robin turns pairwise results into one Elo number per agent (`eval/elo.py`) so
  many agents can be ranked on a single scale.
* With few games the win rate is noisy (10 games has a wide error bar). More games = slower, since
  the engine is CPU-bound.

In [ ]:
net = PokeNet().eval()
print(f"network parameters: {net.num_parameters():,}")

m = play_match(HeuristicAgent(0), RandomAgent(1), games=10)
print(
    f"heuristic vs random: {m.wins}W {m.losses}L {m.draws}D "
    f"({m.games / m.seconds:.1f} games/s on this machine)"
)

rr = round_robin([RandomAgent(0), HeuristicAgent(0), PolicyAgent(net, "untrained-net")], 6)
print(format_report(rr))

## 3. Inside the encoder and the network

`encode` turns one prompt into arrays: global scalars, board slots (active + bench per side), hand,
discards, and one feature row per option plus a final STOP row. The net scores each option row.

**Why** (`model/encoding.py`, `model/net.py`)

* The action space changes size every prompt, so the net doesn't output a fixed list of moves.
  Each option becomes a feature row, the state becomes one vector `h`, and an option's score is a dot
  product between them. Any number of options works with the same weights.
* A final **STOP** row lets multi-pick prompts ("choose up to 3") be answered one pick at a time.
* Illegal options are masked to probability zero, so the policy cannot choose an illegal move.
* Cards are embedded by id *plus* numeric features (HP, type, retreat cost, best damage), so a card
  the net has rarely seen still has meaningful inputs.
* The encoder reads only the acting player's own information. A unit test changes the opponent's
  hidden zones and checks the encoding stays the same (`tests/test_masking.py`).
* Two heads share the trunk: **policy** (which option) and **value** (tanh, -1..1: expected result
  for the player to act). The value head is what lets self-play learn from the game result.
* The net is small (about 2.2M parameters) on purpose: self-play data, not model size, is the limit.

In [ ]:
env = CabtEnv()
d = env.reset(seed=0)
while d is not None and d.num_options < 3:  # find a prompt with a real choice
    d = env.step(list(range(d.min_count)))
enc = encode(d)
for name in ["glob", "slot_ids", "slot_feat", "hand_ids", "opt_feat", "opt_card", "mask"]:
    print(f"{name:10s} {getattr(enc, name).shape}")
print("legal rows:", np.flatnonzero(enc.mask).tolist(), "(last row = STOP)")

priors, value = net.predict([enc])
print("priors:", np.round(priors[0], 3))
print("value (win prob estimate in [-1, 1]):", float(value[0]))
env.close()

In [ ]:
# Where do the parameters live?
for name, p in net.named_parameters():
    if p.numel() > 100_000:
        print(f"{name:40s} {tuple(p.shape)}  {p.numel():,}")

## 4. Edit the config and train

Configs are plain dataclasses (see `configs/default.yaml` and `configs/smoke.yaml`). Anything can be
overridden by dotted key, exactly like `pokezero train a.b=value` on the command line.
The smoke config finishes one tiny iteration in about a minute on CPU. Raise `iterations`,
`selfplay.games_per_iter` and `train.steps_per_iter` for something more meaningful.

Note: self-play uses PUCT search over the engine's forward model, which leaks the opponent's hand
(research-only, see `docs/sdk-findings.md`). The submission agent is the search-free policy net.

**Why** (`train/loop.py`, `train/selfplay.py`, `train/learner.py`, `search/puct.py`)

The loop is AlphaZero in miniature, repeated each iteration:

1. **Self-play:** the current champion plays itself. A tree search (PUCT) looks a few moves ahead,
   guided by the net's priors and value, and picks moves from how often it visited each option.
2. **Targets:** for every decision we store *what the player saw*, *the search's visit
   distribution* (a better policy than the raw net) and *who eventually won*.
3. **Train:** policy loss (cross-entropy toward the search distribution) + value loss (mean squared
   error toward the result) + weight decay. Gradient clipping keeps early updates stable.
4. **Gate:** the candidate plays the champion and only replaces it if it wins at least
   `promote_threshold` (55%). This stops a bad update from poisoning later self-play.
5. **Replay buffer:** a bounded window of recent examples, so old, weaker games age out.

Other choices: chance outcomes (shuffles, coin flips) become separate child nodes; Dirichlet noise at
the root and a temperature for the first 12 moves force exploration; the learner is trained on a
copy, so the champion stays fixed while it generates data.

**Known limit:** the engine's search leaks the opponent's hand, so the targets come from a search with
information a real player lacks. That is why `allow_engine_oracle` must be set explicitly and why the
submission agent is a plain policy net with no search (`agents/az.py`).

**CPU vs GPU:** `train.device` moves only the gradient steps. Games and search are CPU work, so check
`selfplay_seconds` vs `iter_seconds` in the plots below.

In [ ]:
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs").exists())
cfg = load_config(
    REPO / "configs" / "smoke.yaml",
    [
        "run_name=notebook",
        f"artifact_root={tempfile.mkdtemp(prefix='pokezero-runs-')}",
        "iterations=3",
        "selfplay.games_per_iter=2",
        "selfplay.search.simulations=8",
        "train.steps_per_iter=40",
        "eval.games=4",
        "eval.baseline_games=4",
        f"train.device={DEVICE.type}",
    ],
)
print(json.dumps(to_dict(cfg)["train"], indent=2))

In [ ]:
from pokezero.storage import LocalStore
from pokezero.train.loop import run_training

store = LocalStore(Path(cfg.artifact_root) / cfg.run_name)
last = run_training(cfg, store)  # prints one JSON line per iteration

In [ ]:
rows = [json.loads(line) for line in store.get_bytes("metrics.jsonl").decode().splitlines()]
its = [r["step"] for r in rows]
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].plot(its, [r["policy_loss"] for r in rows], marker="o", label="policy")
ax[0].plot(its, [r["value_loss"] for r in rows], marker="o", label="value")
ax[0].set(title="losses", xlabel="iteration")
ax[0].legend()
for k in ("candidate_vs_random", "candidate_vs_heuristic", "h2h_win_rate"):
    ax[1].plot(its, [r[k] for r in rows], marker="o", label=k)
ax[1].set(title="win rates", xlabel="iteration", ylim=(0, 1))
ax[1].legend(fontsize=7)
ax[2].plot(its, [r["iter_seconds"] for r in rows], marker="o", label="iteration")
ax[2].plot(its, [r["selfplay_seconds"] for r in rows], marker="o", label="self-play")
ax[2].set(title="seconds (self-play dominates)", xlabel="iteration")
ax[2].legend()
plt.tight_layout()

In [ ]:
# Load the champion checkpoint and play it against the baselines.
ck = torch.load(store.local_path("champion.pt"), map_location="cpu", weights_only=True)
champ = PokeNet(NetConfig(**ck["net_cfg"]))
champ.load_state_dict(ck["state_dict"])
champ.eval()
m = play_match(PolicyAgent(champ, "champion"), RandomAgent(0), games=10)
print(f"champion (iteration {ck['iteration']}) vs random: {m.win_rate:.0%}")

## 5. Render a replay

Kaggle's own replay viewer, embedded inline. The viewer is a spectator view and shows both hands.
Replays are public on the competition, but don't commit or share these files from the notebook.

**Why** (`viz.py`): replays go through Kaggle's own runner so you see exactly what the competition
shows. The viewer is spectator-only (both hands visible); agents still receive just their own
observation. Watching a game is the fastest way to catch bad play that win rates hide, such as a
net that ends its turn every turn.

In [ ]:
import html as _html

from IPython.display import HTML, display

from pokezero.viz import render_game_html

page, rewards = render_game_html(HeuristicAgent(0), PolicyAgent(champ, "champion"))
print("seat 0 (heuristic) reward:", rewards[0])
display(HTML(f'<iframe srcdoc="{_html.escape(page)}" width="100%" height="700"></iframe>'))

## 6. Decision log (what to say when asked "why?")

| Decision | Choice | Reason | Where |
|---|---|---|---|
| Algorithm | AlphaZero style: self-play + search + policy/value net | No human data needed; improves from its own games | `train/` |
| Action output | Score each option row, mask illegal | Variable number of options per prompt | `model/net.py` |
| Multi-pick prompts | One pick at a time, plus STOP | Keeps the action space small and exact | `model/encoding.py` |
| Hidden information | Encoder sees own hand and public board only | A real player never sees the rest; tested | `tests/test_masking.py` |
| Search | PUCT, chance outcomes as child nodes, off unless `allow_engine_oracle` | Engine search leaks the opponent's hand | `search/puct.py` |
| Submission agent | Policy net only, no search | Leak-free and fast under the 2 vCPU limit | `agents/az.py`, `submission.py` |
| Promotion gate | Candidate must win at least 55% vs champion | Avoid regressions poisoning self-play | `train/loop.py` |
| Reproducibility | Record picks, not seeds | Engine has no seed | `env/base.py` |
| Device | Train on GPU if present, games on CPU | Engine is CPU-only | `train/learner.py` |
| Baselines | Random and heuristic in every report | Detect a broken pipeline early | `agents/`, `eval/` |
| Deck | One fixed starter deck for both sides | Keeps the problem to play, not deckbuilding (assumption to revisit) | `env/cabt.py` |
| Honest status | A few tiny iterations do not make a good player | Expect random-level play until far more self-play | `docs/how-it-works.md` |

## 7. Things to try

* Swap `HeuristicAgent` for your own rules in `pokezero/agents/heuristic.py` and re-run section 2.
* Change `net.*` sizes in the config and compare parameter counts and speed (section 3).
* Compare `train.device=cpu` with `auto` on a GPU runtime and a larger `train.batch_size` /
  `net.hidden`; check `selfplay_seconds` against `iter_seconds` to see where time goes.
* Turn on Weights & Biases with `wandb.enabled=true` (needs `WANDB_API_KEY`).
* Package a checkpoint for submission with `pokezero package --checkpoint <path>`.